### CO2 Emission & Rating Predcition Using Different ML algorithms and Comparing them

--- Data Overview & Preprocessing ---

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,log_loss
from sklearn.preprocessing import StandardScaler

df=pd.read_csv("fuel_data.csv")

# data overview
print("first five rows:\n",df.head(5))
print("\nShape of the data\n:",df.shape)
print("\ncolumns of data:\n",df.columns)
print("\nnull values:\n",df.isnull().sum())
df.info()

# DATA PREPROCESSING

fuel_type_dict = {
    "X": "Regular Gasoline",
    "Z": "Premium Gasoline",
    "D": "Diesel",
    "E": "Ethanol (E85)"
}
df["Fuel type"]=df["Fuel type"].replace(fuel_type_dict)

df_encoded = pd.get_dummies(df, columns=["Vehicle class","Transmission","Fuel type"], drop_first=True)

first five rows:
    Model year   Make              Model                    Vehicle class  \
0        2025  Acura     Integra A-SPEC                        Full-size   
1        2025  Acura     Integra A-SPEC                        Full-size   
2        2025  Acura     Integra Type S                        Full-size   
3        2025  Acura         MDX SH-AWD     Sport utility vehicle: Small   
4        2025  Acura  MDX SH-AWD Type S  Sport utility vehicle: Standard   

   Engine size (L)  Cylinders Transmission Fuel type  City (L/100 km)  \
0              1.5          4          AV7         Z              8.1   
1              1.5          4           M6         Z              8.9   
2              2.0          4           M6         Z             11.1   
3              3.5          6         AS10         Z             12.6   
4              3.0          6         AS10         Z             13.8   

   Highway (L/100 km)  Combined (L/100 km)  Combined (mpg)  \
0                 6.5   

### Linear Regression for CO2 Emission

--- with data leakage ---

In [2]:
X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating"],axis=1)
y=df_encoded["CO2 emissions (g/km)"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

print("\n===== WITH DATA LEAKAGE =====\n")
print("Predicted Score:\n",y_pred)
print("Intercept:",model.intercept_)
print("Coefficient:")
for feat,coef in zip(X,model.coef_):
    print(f"{feat}:{coef:.2f}")

mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)
r2=r2_score(y_test,y_pred)

print("\nMSE:",mse)
print("\nMAE:",mae)
print("\nR2 SCORE:",r2)
print("RMSE:", rmse)


===== WITH DATA LEAKAGE =====

Predicted Score:
 [172.28213732 300.74807685 178.56688001 226.64287223 190.3651364
 248.81115573 189.2133056  223.05066786 214.93094479 328.50109084
 205.89603981 170.89813877 297.41895278 321.74674375 185.04282742
 281.06966763 274.47043946 193.55366816 295.11929022 217.35587096
 258.9251739  184.46202579 220.38421223 251.04171511 330.55452525
 277.84305943 114.78443003 277.09466132 129.6821199  225.59708631
 217.22968659 323.01910179 337.14833284 279.65304043 270.61335352
 140.53171313 206.63191449 161.54758497 140.53171313 268.14921178
 368.74672799 146.04166388 260.82729868 296.50583458 355.78396368
 230.58456483 250.46778465 206.99279804 149.98189702 263.98886539
 203.55351477 188.83344583 321.28312529 162.4212703  225.59708631
 234.79207201 183.21751079 209.68979801 226.37784228 326.05677487
 194.6377583  355.78396368 287.53858811 210.62046204 229.28270496
 385.70486578 145.64215564 382.00162056 340.31515052 220.1368137
 317.69039743 259.21547855 2

by looking at the r2 score and some unrealistic soefficient values i found out there is some data leakage. 
I included "City (L/100 km)","Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)" in the features
But CO₂ emissions are directly calculated from fuel consumption

### Linear Regression for CO2 Emission

--- without data leakage ---

In [3]:
X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating","City (L/100 km)",
                           "Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)"],axis=1)
y=df_encoded["CO2 emissions (g/km)"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

print("\n===== WITHOUT DATA LEAKAGE =====\n")
print("Predicted Score:\n",y_pred)
print("Intercept:",model.intercept_)
print("Coefficient:")
for feat,coef in zip(X,model.coef_):
    print(f"{feat}:{coef:.2f}")

mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)
r2=r2_score(y_test,y_pred)

print("\nMSE:",mse)
print("\nMAE:",mae)
print("\nR2 SCORE:",r2)
print("RMSE:", rmse)


===== WITHOUT DATA LEAKAGE =====

Predicted Score:
 [190.30322312 359.37580166 185.49940509 214.17720378 252.34253796
 271.05007732 185.         230.31078892 195.34427115 321.56441803
 173.93051311 130.92421084 285.00833325 299.09256702 225.36757821
 285.00833325 272.13570509 210.8301859  266.8859296  238.99137198
 249.8218872  205.77743828 259.8025412  267.37677876 286.27072657
 283.6579716  117.23764206 291.85398027 160.56174841 225.86442671
 193.650241   317.76106059 321.56441803 299.53272652 314.94514881
 172.82348029 204.86067845 141.46600229 172.82348029 289.17495339
 366.34892521 139.42392234 284.12296453 250.5725152  330.27381146
 218.53866318 269.88764361 155.30423086 164.06385558 264.85528883
 238.86560537 228.37529156 286.94383062 139.42392234 225.86442671
 236.67185385 138.05190472 185.49940509 252.34253796 338.81426266
 250.5725152  330.27381146 324.76876037 209.60025372 251.66605511
 304.59761812 172.82348029 382.30815435 338.81426266 167.56944818
 324.76876037 252.18424

# CLASSIFICATION MODELS

### Logistic Regression for CO2 rating


In [4]:
X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating","City (L/100 km)",
                           "Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)"],axis=1)
y = (df_encoded["CO2 rating"] >= 5).astype(int)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

print("\nAccuracy:\n",accuracy_score(y_test,y_pred))
print("\nConfusion matrix:\n",confusion_matrix(y_test,y_pred))
print("\n=== Classification Report ===\n",classification_report(y_test,y_pred))

y_prob = model.predict_proba(X_test)
print("\nLog Loss:\n",log_loss(y_test, y_prob))


Accuracy:
 0.8846153846153846

Confusion matrix:
 [[50  8]
 [ 7 65]]

=== Classification Report ===
               precision    recall  f1-score   support

           0       0.88      0.86      0.87        58
           1       0.89      0.90      0.90        72

    accuracy                           0.88       130
   macro avg       0.88      0.88      0.88       130
weighted avg       0.88      0.88      0.88       130


Log Loss:
 0.25542187655767645


### KNN Classification for CO2 Rating

In [5]:
from sklearn.neighbors import KNeighborsClassifier

X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating","City (L/100 km)",
                           "Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)"],axis=1)
y = (df_encoded["CO2 rating"] >= 5).astype(int)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

model=KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled,y_train)
y_pred=model.predict(X_test_scaled)

print("\nAccuracy:\n",accuracy_score(y_test,y_pred))
print("\nConfusion matrix:\n",confusion_matrix(y_test,y_pred))
print("\n=== Classification Report ===\n",classification_report(y_test,y_pred))

y_prob = model.predict_proba(X_test)
print("\nLog Loss:\n",log_loss(y_test, y_prob))


Accuracy:
 0.8384615384615385

Confusion matrix:
 [[42 16]
 [ 5 67]]

=== Classification Report ===
               precision    recall  f1-score   support

           0       0.89      0.72      0.80        58
           1       0.81      0.93      0.86        72

    accuracy                           0.84       130
   macro avg       0.85      0.83      0.83       130
weighted avg       0.85      0.84      0.84       130


Log Loss:
 19.962638800126424


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


### Decision Tree for CO2 Rating

In [12]:
from sklearn.tree import DecisionTreeClassifier

model=DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
model.fit(X_train_scaled,y_train)
y_pred=model.predict(X_test_scaled)

print("\nAccuracy:\n",accuracy_score(y_test,y_pred))
print("\n=== Classification Report ===\n",classification_report(y_test,y_pred))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))


Accuracy:
 0.8615384615384616

=== Classification Report ===
               precision    recall  f1-score   support

           0       0.80      0.91      0.85        58
           1       0.92      0.82      0.87        72

    accuracy                           0.86       130
   macro avg       0.86      0.87      0.86       130
weighted avg       0.87      0.86      0.86       130

Confusion Matrix:
 [[53  5]
 [13 59]]


In [7]:
from sklearn.naive_bayes import GaussianNB

model_nb=GaussianNB()
model_nb.fit(X_train,y_train)
y_pred_nb=model_nb.predict(X_test)
print("\nPredicted Values:\n",y_pred_nb)


Predicted Values:
 [0 0 1 1 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 1 0
 1 1 0 0 1 0 0 0 0 0 1 1 0 1 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 1 0 0 1 0 1 0 1
 0 0 0 1 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1]


In [8]:
print("Accuracy:",accuracy_score(y_test,y_pred_nb))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred_nb))
print("Classification Report:\n",classification_report(y_test,y_pred_nb))

Accuracy: 0.676923076923077
Confusion Matrix:
 [[57  1]
 [41 31]]
Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.98      0.73        58
           1       0.97      0.43      0.60        72

    accuracy                           0.68       130
   macro avg       0.78      0.71      0.66       130
weighted avg       0.80      0.68      0.66       130



In [11]:
from sklearn.svm import SVC

model_svc=SVC(kernel='rbf',class_weight="balanced")
model_svc.fit(X_train_scaled,y_train)
y_pred_svc=model_svc.predict(X_test_scaled)
print("Predicted Values are:\n",y_pred_svc)
print("\nAccuracy:\n",accuracy_score(y_test,y_pred_svc))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,y_pred_svc))
print("\nClassification Report:\n",classification_report(y_test,y_pred_svc))

Predicted Values are:
 [1 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 0 1 0 1 1 1 1 0 0 1 1 0 1 1 1 0 0 0 0 1 1
 1 1 0 0 1 0 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 0 0 0 0 0 1 1 0 1 0 0 1 0 1 1 1
 0 0 1 1 1 1 0 1 0 1 1 1 1 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0
 1 0 0 0 1 0 1 1 1 0 0 1 0 0 1 0 0 0 1]

Accuracy:
 0.8461538461538461

Confusion Matrix:
 [[53  5]
 [15 57]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.91      0.84        58
           1       0.92      0.79      0.85        72

    accuracy                           0.85       130
   macro avg       0.85      0.85      0.85       130
weighted avg       0.86      0.85      0.85       130

